<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/CREST_TS_Conformer_Puckering_Analysis_Cycloetherification_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-

"""
============================================================
TS RING-PUCKER ANALYSIS
CREST + Cremer-Pople + Six-Ring-Torsion Deduplication
============================================================

Purpose
-------
1. Run constrained CREST on each input XYZ structure.
2. Preserve the forming O17-C22 distance using a CREST constraint.
3. Calculate Cremer-Pople puckering parameters.
4. Calculate six sequential ring torsions.
5. Remove conformers that have the same six-torsion pattern
   within the specified tolerance.
6. Keep the lowest-energy conformer for each unique puckering class.
7. Process all XYZ structures with checkpoint/resume support.

IMPORTANT
---------
CREST structures are constrained conformers/minima.
They are NOT verified transition states.

Use retained structures as starting geometries for
subsequent TS optimization and frequency calculations.
"""


# ============================================================
# BLOCK 1 — INSTALL CREST/xTB
# ============================================================

!apt-get -qq update
!apt-get -qq install -y xtb

print("Checking xTB:")
!xtb --version

print("\nInstalling CREST:")
!wget -q -O /content/crest.tar.xz \
    "https://github.com/crest-lab/crest/releases/download/latest/crest-gnu-12-ubuntu-latest.tar.xz"

!rm -rf /content/crest
!tar -xf /content/crest.tar.xz -C /content
!chmod +x /content/crest/crest

print("\nChecking CREST:")
!/content/crest/crest --version


# ============================================================
# BLOCK 2 — IMPORTS + GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import time

import numpy as np
import pandas as pd

print("\nPython environment ready.")


# ============================================================
# BLOCK 3 — USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# Main input folder
# ------------------------------------------------------------

DRIVE_FOLDER = Path(
    "/content/drive/MyDrive/Cycloetherification_xyz_files"
)

# All persistent results will be saved here.
RESULTS_FOLDER = DRIVE_FOLDER / "CREST_results"
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Ring definition
# ------------------------------------------------------------

# Six ring atoms in connectivity order.
# Atom numbering is 1-based, as used in your XYZ files.
ring_atoms = [17, 18, 19, 20, 21, 22]


# ------------------------------------------------------------
# Forming bond
# ------------------------------------------------------------

# Bond constrained during CREST.
forming_bond = [17, 22]


# ------------------------------------------------------------
# Constraint force constant
# ------------------------------------------------------------

fc = 1.0


# ------------------------------------------------------------
# Single-structure CREST settings
# ------------------------------------------------------------

method = "gfn2"
quick_mode = True
threads = 2


# ------------------------------------------------------------
# Torsion-based deduplication
# ------------------------------------------------------------

# Two conformers are considered the same ring-pucker family
# when ALL SIX torsions differ by <= this value.
torsion_tolerance_deg = 5.0


# ------------------------------------------------------------
# Batch CREST settings
# ------------------------------------------------------------

batch_method = "gff"
batch_quick_mode = True
batch_threads = 2
batch_timeout_seconds = 1800


# ------------------------------------------------------------
# Optional per-file atom-number overrides
# ------------------------------------------------------------

# Normally leave this empty.
#
# Example:
#
# OVERRIDES = {
#     "RR_example.xyz": {
#         "ring_atoms": [17, 18, 19, 20, 21, 22],
#         "forming_bond": [17, 22],
#     }
# }

OVERRIDES = {}


# ------------------------------------------------------------
# CREST executable
# ------------------------------------------------------------

CREST_EXECUTABLE = "/content/crest/crest"


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

if not DRIVE_FOLDER.exists():
    raise FileNotFoundError(
        f"Input folder does not exist:\n{DRIVE_FOLDER}"
    )

if not Path(CREST_EXECUTABLE).exists():
    raise FileNotFoundError(
        f"CREST executable not found:\n{CREST_EXECUTABLE}"
    )

if len(ring_atoms) != 6:
    raise ValueError(
        "ring_atoms must contain exactly six atoms."
    )

if len(forming_bond) != 2:
    raise ValueError(
        "forming_bond must contain exactly two atoms."
    )

if method not in {"gfn2", "gff"}:
    raise ValueError(
        "method must be 'gfn2' or 'gff'."
    )

if batch_method not in {"gfn2", "gff"}:
    raise ValueError(
        "batch_method must be 'gfn2' or 'gff'."
    )


print("\n============================================================")
print("USER SETTINGS")
print("============================================================")
print("Input folder       :", DRIVE_FOLDER)
print("Results folder     :", RESULTS_FOLDER)
print("Ring atoms         :", ring_atoms)
print("Forming bond       :", forming_bond)
print("Force constant     :", fc)
print("Batch method       :", batch_method)
print("Batch quick mode   :", batch_quick_mode)
print("Batch threads      :", batch_threads)
print("Batch timeout (s)  :", batch_timeout_seconds)
print("Torsion tolerance  :", torsion_tolerance_deg, "degrees")


# ============================================================
# BLOCK 4 — XYZ FUNCTIONS
# ============================================================

def parse_energy_from_comment(comment):
    """
    Extract an energy from an XYZ comment line.

    CREST comment formats can vary, so several patterns
    are checked.
    """

    patterns = [
        r"energy\s*=\s*([-+]?\d+(?:\.\d*)?(?:[Ee][-+]?\d+)?)",
        r"\bE\s*=\s*([-+]?\d+(?:\.\d*)?(?:[Ee][-+]?\d+)?)",
        r"([-+]?\d+\.\d+(?:[Ee][-+]?\d+)?)",
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            comment,
            flags=re.IGNORECASE
        )

        if match:
            return float(match.group(1))

    return np.nan


def read_xyz_frames(path):
    """
    Read one or more XYZ structures.

    Returns
    -------
    list of dictionaries

    Each dictionary contains:
        atoms
        coords
        comment
        energy_Eh
    """

    path = Path(path)

    with path.open("r") as f:
        lines = f.readlines()

    frames = []
    position = 0

    while position < len(lines):

        # Skip blank lines.
        if not lines[position].strip():
            position += 1
            continue

        # Number of atoms.
        try:
            n_atoms = int(lines[position].strip())
        except ValueError as exc:
            raise ValueError(
                f"Invalid atom-count line at line "
                f"{position + 1} in {path}"
            ) from exc

        # XYZ frame length.
        end = position + n_atoms + 2

        if end > len(lines):
            raise ValueError(
                f"Incomplete XYZ frame in {path}"
            )

        comment = lines[position + 1].rstrip("\n")

        atom_lines = lines[position + 2:end]

        atoms = []
        coords = []

        for line_number, line in enumerate(
            atom_lines,
            start=position + 3
        ):

            parts = line.split()

            if len(parts) < 4:
                raise ValueError(
                    f"Invalid XYZ line {line_number} "
                    f"in {path}: {line}"
                )

            atoms.append(parts[0])

            coords.append([
                float(parts[1]),
                float(parts[2]),
                float(parts[3]),
            ])

        frames.append({
            "atoms": atoms,
            "coords": np.asarray(
                coords,
                dtype=float
            ),
            "comment": comment,
            "energy_Eh": parse_energy_from_comment(
                comment
            ),
        })

        position = end

    return frames


def write_xyz_frames(frames, path):
    """
    Write one or more XYZ structures.
    """

    path = Path(path)

    with path.open("w") as f:

        for frame in frames:

            atoms = frame["atoms"]
            coords = frame["coords"]
            comment = frame.get("comment", "")

            f.write(f"{len(atoms)}\n")
            f.write(f"{comment}\n")

            for element, xyz in zip(atoms, coords):

                f.write(
                    f"{element:2s} "
                    f"{xyz[0]: .10f} "
                    f"{xyz[1]: .10f} "
                    f"{xyz[2]: .10f}\n"
                )


def validate_atom_indices(frame, indices, name):
    """
    Validate 1-based atom numbers.
    """

    n_atoms = len(frame["atoms"])

    if not indices:
        raise ValueError(
            f"{name} cannot be empty."
        )

    if len(set(indices)) != len(indices):
        raise ValueError(
            f"{name} contains duplicate atom indices."
        )

    if any(
        i < 1 or i > n_atoms
        for i in indices
    ):
        raise IndexError(
            f"{name}={indices} is invalid for "
            f"a structure with {n_atoms} atoms."
        )


def distance_from_frame(frame, i, j):
    """
    Calculate distance between two 1-based atoms in Å.
    """

    validate_atom_indices(
        frame,
        [i, j],
        "distance_atoms"
    )

    coords = frame["coords"]

    return float(
        np.linalg.norm(
            coords[i - 1] - coords[j - 1]
        )
    )


# ============================================================
# BLOCK 5 — RING / CREMER–POPLE / TORSION FUNCTIONS
# ============================================================

def cremer_pople_6(ring_coords):
    """
    Calculate Cremer-Pople Q, theta and phi
    for a six-membered ring.

    ring_coords must contain six atoms in
    connectivity order.
    """

    R = np.asarray(
        ring_coords,
        dtype=float
    )

    if R.shape != (6, 3):
        raise ValueError(
            "Expected ring coordinates with shape "
            f"(6,3), got {R.shape}"
        )

    # Center the ring.
    centered = R - R.mean(axis=0)

    # Best-fit ring plane.
    _, _, vh = np.linalg.svd(
        centered,
        full_matrices=False
    )

    normal = vh[-1]

    # Out-of-plane coordinates.
    z = centered @ normal

    j = np.arange(6)

    # q2 cosine component.
    c = np.sqrt(2.0 / 6.0) * np.sum(
        z * np.cos(
            2.0 * np.pi * j / 3.0
        )
    )

    # q2 sine component.
    s = -np.sqrt(2.0 / 6.0) * np.sum(
        z * np.sin(
            2.0 * np.pi * j / 3.0
        )
    )

    q2 = np.sqrt(
        c**2 + s**2
    )

    # q3 component.
    q3 = np.sqrt(1.0 / 6.0) * np.sum(
        z * ((-1.0) ** j)
    )

    # Total puckering amplitude.
    Q = float(
        np.sqrt(
            q2**2 + q3**2
        )
    )

    if Q < 1e-12:
        return 0.0, 0.0, 0.0

    theta = float(
        np.degrees(
            np.arccos(
                np.clip(
                    q3 / Q,
                    -1.0,
                    1.0
                )
            )
        )
    )

    phi = float(
        np.degrees(
            np.arctan2(s, c)
        ) % 360.0
    )

    return Q, theta, phi


def classify_6ring(theta, phi):
    """
    Approximate six-membered-ring pucker classification.

    Classification is heuristic and should be interpreted
    together with the actual Q/theta/phi values.
    """

    theta = float(theta)
    phi = float(phi) % 360.0

    # Chair region.
    if theta < 15.0 or theta > 165.0:
        return "Chair (C)"

    # Boat / twist-boat region.
    if 75.0 <= theta <= 105.0:

        m = phi % 60.0

        if m < 15.0 or m > 45.0:
            return "Boat (B)"

        return "Twist-boat/Skew (S)"

    # Envelope / half-chair region.
    if (
        35.0 <= theta <= 65.0
        or
        115.0 <= theta <= 145.0
    ):

        m = phi % 60.0

        if m < 15.0 or m > 45.0:
            return "Envelope (E)"

        return "Half-chair (H)"

    return "Intermediate"


def dihedral_angle(coords, i, j, k, l):
    """
    Calculate a dihedral angle in degrees.

    Atom indices here are 0-based.
    """

    p0, p1, p2, p3 = coords[
        [i, j, k, l]
    ]

    b0 = -(p1 - p0)
    b1 = p2 - p1
    b2 = p3 - p2

    b1_norm = np.linalg.norm(b1)

    if b1_norm < 1e-12:
        raise ValueError(
            "Central bond has zero length."
        )

    b1 = b1 / b1_norm

    v = (
        b0
        - np.dot(b0, b1) * b1
    )

    w = (
        b2
        - np.dot(b2, b1) * b1
    )

    if np.linalg.norm(v) < 1e-12:
        raise ValueError(
            "First projected vector is too small."
        )

    if np.linalg.norm(w) < 1e-12:
        raise ValueError(
            "Second projected vector is too small."
        )

    x = np.dot(v, w)

    y = np.dot(
        np.cross(b1, v),
        w
    )

    return float(
        np.degrees(
            np.arctan2(y, x)
        )
    )


def ring_torsions(frame, ring_atoms):
    """
    Calculate six sequential ring torsions.

    For:
        [a0,a1,a2,a3,a4,a5]

    calculate:

        a0-a1-a2-a3
        a1-a2-a3-a4
        a2-a3-a4-a5
        a3-a4-a5-a0
        a4-a5-a0-a1
        a5-a0-a1-a2
    """

    if len(ring_atoms) != 6:
        raise ValueError(
            "ring_atoms must contain six atoms."
        )

    validate_atom_indices(
        frame,
        ring_atoms,
        "ring_atoms"
    )

    indices = [
        atom - 1
        for atom in ring_atoms
    ]

    coords = frame["coords"]

    torsions = []

    for shift in range(6):

        quartet = [
            indices[
                (shift + offset) % 6
            ]
            for offset in range(4)
        ]

        torsions.append(
            dihedral_angle(
                coords,
                *quartet
            )
        )

    return np.asarray(
        torsions,
        dtype=float
    )


def torsion_differences(
    torsions_a,
    torsions_b
):
    """
    Calculate smallest periodic absolute
    differences between two torsion vectors.
    """

    a = np.asarray(
        torsions_a,
        dtype=float
    )

    b = np.asarray(
        torsions_b,
        dtype=float
    )

    if a.shape != b.shape:
        raise ValueError(
            "Torsion vectors have different shapes."
        )

    return np.abs(
        (a - b + 180.0) % 360.0
        - 180.0
    )


# ============================================================
# BLOCK 6 — CREST EXECUTION
# ============================================================

def write_constraint_file(
    workdir,
    forming_bond,
    fc
):
    """
    Write the CREST constraint input.
    """

    workdir = Path(workdir)

    workdir.mkdir(
        parents=True,
        exist_ok=True
    )

    constraint_text = (
        "$constrain\n"
        f"distance: "
        f"{forming_bond[0]}, "
        f"{forming_bond[1]}, auto\n"
        f"force constant={fc}\n"
        "$end\n"
    )

    path = workdir / "constraints.inp"

    path.write_text(
        constraint_text
    )

    return path


def run_crest_live(
    xyz_file,
    workdir,
    forming_bond,
    fc=1.0,
    method="gfn2",
    quick_mode=True,
    threads=2,
    timeout_seconds=1800
):
    """
    Run one constrained CREST calculation.

    stdout is printed live and saved.
    stderr is also saved.
    """

    xyz_file = Path(xyz_file)
    workdir = Path(workdir)

    if not xyz_file.exists():
        raise FileNotFoundError(
            f"XYZ file not found:\n{xyz_file}"
        )

    if method not in {"gfn2", "gff"}:
        raise ValueError(
            "method must be 'gfn2' or 'gff'."
        )

    if int(threads) < 1:
        raise ValueError(
            "threads must be >= 1."
        )

    if (
        timeout_seconds is not None
        and float(timeout_seconds) <= 0
    ):
        raise ValueError(
            "timeout_seconds must be positive."
        )

    # Start from a clean temporary directory.
    if workdir.exists():
        shutil.rmtree(workdir)

    workdir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Read input structure.
    input_frames = read_xyz_frames(
        xyz_file
    )

    if not input_frames:
        raise ValueError(
            f"No structure found in {xyz_file}"
        )

    validate_atom_indices(
        input_frames[0],
        forming_bond,
        "forming_bond"
    )

    # CREST expects the structure in its working directory.
    shutil.copy2(
        xyz_file,
        workdir / "struc.xyz"
    )

    # Write constraint.
    constraint_path = write_constraint_file(
        workdir,
        forming_bond,
        fc
    )

    # Build CREST command.
    cmd = [
        CREST_EXECUTABLE,
        "struc.xyz",
        "--cinp",
        constraint_path.name,
        "--T",
        str(int(threads)),
    ]

    if method == "gff":
        cmd.append("--gfnff")

    if quick_mode:
        cmd.append("-mquick")

    print("\nCREST command:")
    print(" ".join(cmd))
    print()

    stdout_path = (
        workdir / "crest_stdout.log"
    )

    stderr_path = (
        workdir / "crest_stderr.log"
    )

    start_time = time.time()

    process = subprocess.Popen(
        cmd,
        cwd=workdir,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )

    stdout_lines = []
    stderr_text = ""
    timed_out = False

    try:

        while True:

            line = (
                process.stdout.readline()
                if process.stdout
                else ""
            )

            if line:

                print(
                    line,
                    end="",
                    flush=True
                )

                stdout_lines.append(
                    line
                )

            if process.poll() is not None:
                break

            if (
                timeout_seconds is not None
                and
                time.time() - start_time
                > float(timeout_seconds)
            ):

                timed_out = True

                process.kill()

                break

            if not line:
                time.sleep(0.1)

        # Collect remaining output.
        remaining_stdout = (
            process.stdout.read()
            if process.stdout
            else ""
        )

        if remaining_stdout:

            print(
                remaining_stdout,
                end="",
                flush=True
            )

            stdout_lines.append(
                remaining_stdout
            )

        stderr_text = (
            process.stderr.read()
            if process.stderr
            else ""
        )

        return_code = process.wait()

    finally:

        if process.poll() is None:

            process.kill()
            process.wait()

    # Save logs.
    stdout_path.write_text(
        "".join(stdout_lines)
    )

    stderr_path.write_text(
        stderr_text
    )

    elapsed = (
        time.time()
        - start_time
    )

    print()
    print(
        f"CREST runtime: {elapsed:.1f} s"
    )

    print(
        f"Return code: {return_code}"
    )

    if timed_out:

        raise TimeoutError(
            f"CREST exceeded "
            f"{timeout_seconds} seconds for "
            f"{xyz_file.name}"
        )

    if return_code != 0:

        raise RuntimeError(
            f"CREST failed for "
            f"{xyz_file.name}\n\n"
            f"Working directory:\n"
            f"{workdir}\n\n"
            f"Return code: {return_code}\n\n"
            f"Last stderr:\n"
            f"{stderr_text[-3000:]}"
        )

    conformer_file = (
        workdir / "crest_conformers.xyz"
    )

    if not conformer_file.exists():

        raise FileNotFoundError(
            "CREST finished but did not create:\n"
            f"{conformer_file}"
        )

    if conformer_file.stat().st_size == 0:

        raise ValueError(
            f"CREST output is empty:\n"
            f"{conformer_file}"
        )

    return conformer_file


# Backward-compatible wrapper.
def run_crest(*args, **kwargs):
    return run_crest_live(
        *args,
        **kwargs
    )


# ============================================================
# BLOCK 7 — CONFORMER ANALYSIS
# ============================================================

def analyze_conformers(
    conformer_file,
    ring_atoms,
    forming_bond
):
    """
    Calculate structural descriptors for every CREST conformer.

    Returns
    -------
    list of dictionaries
    """

    frames = read_xyz_frames(
        conformer_file
    )

    if not frames:
        raise ValueError(
            f"No conformers found in "
            f"{conformer_file}"
        )

    records = []

    for original_index, frame in enumerate(
        frames,
        start=1
    ):

        validate_atom_indices(
            frame,
            ring_atoms,
            "ring_atoms"
        )

        validate_atom_indices(
            frame,
            forming_bond,
            "forming_bond"
        )

        # Ring coordinates.
        ring_coords = frame["coords"][
            [
                atom - 1
                for atom in ring_atoms
            ]
        ]

        # Cremer-Pople.
        Q, theta, phi = cremer_pople_6(
            ring_coords
        )

        # Six torsions.
        torsions = ring_torsions(
            frame,
            ring_atoms
        )

        # Forming bond.
        bond_distance = distance_from_frame(
            frame,
            forming_bond[0],
            forming_bond[1]
        )

        record = {
            "original_conformer":
                original_index,

            "energy_Eh":
                frame["energy_Eh"],

            "forming_bond_A":
                bond_distance,

            "Q_A":
                Q,

            "theta_deg":
                theta,

            "phi_deg":
                phi,

            "classification":
                classify_6ring(
                    theta,
                    phi
                ),
        }

        for i, torsion in enumerate(
            torsions,
            start=1
        ):

            record[
                f"torsion_{i}_deg"
            ] = torsion

        records.append({
            "frame": frame,
            "torsions": torsions,
            "record": record
        })

    return records


# ============================================================
# BLOCK 8 — TORSION DEDUPLICATION
# ============================================================

def deduplicate_by_six_torsions(
    analyzed_records,
    tolerance_deg=5.0
):
    """
    Sort conformers by energy and retain the lowest-energy
    conformer from each unique six-torsion family.

    Two conformers are considered duplicates when ALL SIX
    torsion differences are <= tolerance_deg.
    """

    # --------------------------------------------------------
    # Sort by energy.
    # Finite energies first, lowest energy first.
    # --------------------------------------------------------

    analyzed_records = sorted(
        analyzed_records,
        key=lambda item: (
            not np.isfinite(
                item["record"]["energy_Eh"]
            ),
            item["record"]["energy_Eh"]
            if np.isfinite(
                item["record"]["energy_Eh"]
            )
            else np.inf
        )
    )

    retained = []
    duplicates = []

    # --------------------------------------------------------
    # Compare every conformer with retained conformers.
    # --------------------------------------------------------

    for item in analyzed_records:

        frame = item["frame"]
        torsions = item["torsions"]
        record = item["record"]

        duplicate_of = None
        duplicate_diff = None

        for retained_item in retained:

            differences = torsion_differences(
                torsions,
                retained_item["torsions"]
            )

            if np.all(
                differences <= tolerance_deg
            ):

                duplicate_of = (
                    retained_item[
                        "retained_conformer"
                    ]
                )

                duplicate_diff = differences

                break

        # ----------------------------------------------------
        # New unique torsion family.
        # ----------------------------------------------------

        if duplicate_of is None:

            retained_number = (
                len(retained) + 1
            )

            retained_item = {
                "frame": frame,
                "torsions": torsions,
                "retained_conformer":
                    retained_number
            }

            retained.append(
                retained_item
            )

            record.update({

                "retained_conformer":
                    retained_number,

                "status":
                    "retained",

                "duplicate_of":
                    np.nan,

                "max_torsion_difference_deg":
                    0.0,

                "mean_torsion_difference_deg":
                    0.0,
            })

        # ----------------------------------------------------
        # Duplicate torsion family.
        # ----------------------------------------------------

        else:

            record.update({

                "retained_conformer":
                    np.nan,

                "status":
                    "removed_torsion_duplicate",

                "duplicate_of":
                    duplicate_of,

                "max_torsion_difference_deg":
                    float(
                        np.max(
                            duplicate_diff
                        )
                    ),

                "mean_torsion_difference_deg":
                    float(
                        np.mean(
                            duplicate_diff
                        )
                    ),
            })

            duplicates.append(
                record.copy()
            )

        # Store retained record separately.
        if duplicate_of is None:

            retained_item["record"] = (
                record.copy()
            )

    # --------------------------------------------------------
    # Create data tables.
    # --------------------------------------------------------

    retained_records = [
        item["record"]
        for item in retained
    ]

    retained_df = pd.DataFrame(
        retained_records
    )

    duplicates_df = pd.DataFrame(
        duplicates
    )

    # --------------------------------------------------------
    # Relative energy.
    # --------------------------------------------------------

    if len(retained_df):

        minimum_energy = (
            retained_df["energy_Eh"].min()
        )

        if np.isfinite(
            minimum_energy
        ):

            retained_df["rel_kcal"] = (
                (
                    retained_df["energy_Eh"]
                    - minimum_energy
                )
                * 627.5095
            )

        else:

            retained_df["rel_kcal"] = np.nan

        # Sort retained structures by energy.
        order = np.argsort(
            retained_df[
                "rel_kcal"
            ].fillna(np.inf).values
        )

        retained_df = (
            retained_df.iloc[order]
            .reset_index(drop=True)
        )

        # IMPORTANT:
        # retained frames must follow the same order
        # as retained_df.
        retained_lookup = {
            int(item["record"][
                "retained_conformer"
            ]): item
            for item in retained
        }

        sorted_retained = []

        for retained_number in (
            retained_df[
                "retained_conformer"
            ].astype(int)
        ):

            sorted_retained.append(
                retained_lookup[
                    retained_number
                ]
            )

        retained = sorted_retained

    return (
        retained,
        retained_df,
        duplicates_df
    )


# ============================================================
# BLOCK 9 — PROCESS ONE STRUCTURE
# ============================================================

def process_one_structure(
    xyz_file,
    results_folder,
    ring_atoms,
    forming_bond,
    fc=1.0,
    method="gff",
    quick_mode=True,
    threads=2,
    torsion_tolerance_deg=5.0,
    timeout_seconds=1800
):
    """
    Run CREST, analyze conformers, deduplicate by torsions,
    and save all results for one input structure.
    """

    xyz_file = Path(xyz_file)
    results_folder = Path(
        results_folder
    )

    tag = xyz_file.stem

    # Temporary CREST working directory.
    workdir = (
        Path("/content")
        / f"crest_run_{tag}"
    )

    # Permanent output directory.
    output_dir = (
        results_folder
        / f"{tag}_CREST_conformations"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print("\n")
    print("=" * 80)
    print(f"PROCESSING: {tag}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Run CREST.
    # --------------------------------------------------------

    conformer_file = run_crest_live(
        xyz_file=xyz_file,
        workdir=workdir,
        forming_bond=forming_bond,
        fc=fc,
        method=method,
        quick_mode=quick_mode,
        threads=threads,
        timeout_seconds=timeout_seconds
    )

    # --------------------------------------------------------
    # 2. Analyze all CREST conformers.
    # --------------------------------------------------------

    analyzed = analyze_conformers(
        conformer_file=conformer_file,
        ring_atoms=ring_atoms,
        forming_bond=forming_bond
    )

    raw_count = len(analyzed)

    # --------------------------------------------------------
    # 3. Deduplicate.
    # --------------------------------------------------------

    (
        retained,
        retained_df,
        duplicates_df
    ) = deduplicate_by_six_torsions(
        analyzed,
        tolerance_deg=
            torsion_tolerance_deg
    )

    # --------------------------------------------------------
    # 4. Preserve raw CREST ensemble.
    # --------------------------------------------------------

    shutil.copy2(
        conformer_file,
        output_dir
        / f"{tag}_raw_crest_conformers.xyz"
    )

    # --------------------------------------------------------
    # 5. Save individual retained conformers.
    # --------------------------------------------------------

    for item in retained:

        frame = dict(
            item["frame"]
        )

        retained_number = int(
            item["record"][
                "retained_conformer"
            ]
        )

        energy = item["record"][
            "energy_Eh"
        ]

        rel_energy = item["record"].get(
            "rel_kcal",
            np.nan
        )

        frame["comment"] = (
            f"retained_conformer="
            f"{retained_number} "
            f"energy_Eh="
            f"{energy:.12f} "
            f"rel_kcal="
            f"{rel_energy:.6f}"
        )

        output_path = (
            output_dir
            / f"{tag}_retained_"
            f"{retained_number:03d}.xyz"
        )

        write_xyz_frames(
            [frame],
            output_path
        )

    # --------------------------------------------------------
    # 6. Save retained ensemble.
    # --------------------------------------------------------

    retained_frames = [
        item["frame"]
        for item in retained
    ]

    retained_ensemble_path = (
        output_dir
        / f"{tag}_retained_ensemble.xyz"
    )

    # Always create the file.
    write_xyz_frames(
        retained_frames,
        retained_ensemble_path
    )

    # --------------------------------------------------------
    # 7. Save CSV tables.
    # --------------------------------------------------------

    retained_csv = (
        output_dir
        / f"{tag}_retained_pucker_results.csv"
    )

    duplicates_csv = (
        output_dir
        / f"{tag}_torsion_duplicates.csv"
    )

    retained_df.to_csv(
        retained_csv,
        index=False
    )

    duplicates_df.to_csv(
        duplicates_csv,
        index=False
    )

    # --------------------------------------------------------
    # 8. Save metadata.
    # --------------------------------------------------------

    metadata = {

        "source_file":
            str(xyz_file),

        "ring_atoms":
            list(ring_atoms),

        "forming_bond":
            list(forming_bond),

        "constraint_force_constant":
            fc,

        "method":
            method,

        "quick_mode":
            quick_mode,

        "threads":
            threads,

        "torsion_tolerance_deg":
            torsion_tolerance_deg,

        "raw_conformer_count":
            raw_count,

        "retained_conformer_count":
            len(retained),

        "removed_torsion_duplicate_count":
            len(duplicates_df),
    }

    metadata_path = (
        output_dir
        / f"{tag}_metadata.json"
    )

    with metadata_path.open(
        "w"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2
        )

    print()
    print(
        f"{tag}: "
        f"{raw_count} CREST conformers"
    )

    print(
        f"{tag}: "
        f"{len(retained)} retained"
    )

    print(
        f"{tag}: "
        f"{len(duplicates_df)} torsion duplicates removed"
    )

    return (
        retained_df,
        duplicates_df
    )


# ============================================================
# BLOCK 10 — VALIDATE INPUT FILES
# ============================================================

xyz_files = sorted(
    DRIVE_FOLDER.glob("*.xyz")
)

print("\n")
print("=" * 80)
print("INPUT VALIDATION")
print("=" * 80)

print(
    f"Found {len(xyz_files)} XYZ files."
)

if not xyz_files:

    raise FileNotFoundError(
        f"No XYZ files found in:\n"
        f"{DRIVE_FOLDER}"
    )


validation_records = []


for xyz_file in xyz_files:

    config = {

        "ring_atoms":
            list(ring_atoms),

        "forming_bond":
            list(forming_bond),
    }

    config.update(
        OVERRIDES.get(
            xyz_file.name,
            {}
        )
    )

    try:

        frames = read_xyz_frames(
            xyz_file
        )

        if not frames:

            raise ValueError(
                "No XYZ frames found."
            )

        first_frame = frames[0]

        validate_atom_indices(
            first_frame,
            config["ring_atoms"],
            "ring_atoms"
        )

        validate_atom_indices(
            first_frame,
            config["forming_bond"],
            "forming_bond"
        )

        initial_distance = (
            distance_from_frame(
                first_frame,
                config["forming_bond"][0],
                config["forming_bond"][1]
            )
        )

        validation_records.append({

            "structure":
                xyz_file.name,

            "n_atoms":
                len(first_frame["atoms"]),

            "input_frames":
                len(frames),

            "initial_forming_bond_A":
                initial_distance,

            "status":
                "OK",

            "error":
                "",
        })

    except Exception as exc:

        validation_records.append({

            "structure":
                xyz_file.name,

            "n_atoms":
                np.nan,

            "input_frames":
                np.nan,

            "initial_forming_bond_A":
                np.nan,

            "status":
                "FAILED",

            "error":
                repr(exc),
        })


validation_df = pd.DataFrame(
    validation_records
)

display(validation_df)


validation_path = (
    RESULTS_FOLDER
    / "INPUT_validation.csv"
)

validation_df.to_csv(
    validation_path,
    index=False
)


if (
    validation_df["status"]
    != "OK"
).any():

    raise RuntimeError(
        "Input validation failed.\n\n"
        "Check:\n"
        f"{validation_path}\n\n"
        "Fix the input files before "
        "running the batch."
    )


print(
    "\nAll input XYZ files passed validation."
)


# ============================================================
# BLOCK 11 — BATCH PROCESSING + CHECKPOINT / RESUME
# ============================================================

def structure_output_dir(
    results_folder,
    tag
):
    """
    Return permanent output directory
    for one structure.
    """

    return (
        Path(results_folder)
        / f"{tag}_CREST_conformations"
    )


def structure_is_complete(
    results_folder,
    tag
):
    """
    Check whether all important output files
    already exist.

    If yes, the structure will NOT be recalculated.
    """

    output_dir = structure_output_dir(
        results_folder,
        tag
    )

    required_files = [

        output_dir
        / f"{tag}_raw_crest_conformers.xyz",

        output_dir
        / f"{tag}_retained_ensemble.xyz",

        output_dir
        / f"{tag}_retained_pucker_results.csv",

        output_dir
        / f"{tag}_torsion_duplicates.csv",

        output_dir
        / f"{tag}_metadata.json",
    ]

    return all(
        path.exists()
        and path.stat().st_size > 0
        for path in required_files
    )


def load_completed_structure(
    results_folder,
    tag
):
    """
    Load previously completed CSV results.
    """

    output_dir = structure_output_dir(
        results_folder,
        tag
    )

    retained_df = pd.read_csv(
        output_dir
        / f"{tag}_retained_pucker_results.csv"
    )

    duplicates_df = pd.read_csv(
        output_dir
        / f"{tag}_torsion_duplicates.csv"
    )

    return (
        retained_df,
        duplicates_df
    )


# ------------------------------------------------------------
# Batch output files
# ------------------------------------------------------------

progress_path = (
    RESULTS_FOLDER
    / "BATCH_progress.csv"
)

failure_path = (
    RESULTS_FOLDER
    / "BATCH_failures.csv"
)

retained_path = (
    RESULTS_FOLDER
    / "ALL_retained_pucker_results.csv"
)

duplicates_path = (
    RESULTS_FOLDER
    / "ALL_torsion_duplicates.csv"
)


# ------------------------------------------------------------
# Containers for results
# ------------------------------------------------------------

all_retained = []
all_duplicates = []

completed_records = []
failure_records = []


print("\n")
print("=" * 80)
print("STARTING CREST BATCH")
print("=" * 80)


# ------------------------------------------------------------
# Main batch loop
# ------------------------------------------------------------

for file_number, xyz_file in enumerate(
    xyz_files,
    start=1
):

    tag = xyz_file.stem

    print("\n")
    print("=" * 80)

    print(
        f"[{file_number}/{len(xyz_files)}] "
        f"Processing: {xyz_file.name}"
    )

    print("=" * 80)


    # --------------------------------------------------------
    # File-specific configuration.
    # --------------------------------------------------------

    config = {

        "ring_atoms":
            list(ring_atoms),

        "forming_bond":
            list(forming_bond),
    }

    config.update(
        OVERRIDES.get(
            xyz_file.name,
            {}
        )
    )


    try:

        # ----------------------------------------------------
        # RESUME CHECK
        # ----------------------------------------------------

        if structure_is_complete(
            RESULTS_FOLDER,
            tag
        ):

            print()
            print(
                "Already complete."
            )

            print(
                "Loading saved results:"
                f" {tag}"
            )

            retained_df, duplicates_df = (
                load_completed_structure(
                    RESULTS_FOLDER,
                    tag
                )
            )

        else:

            # ------------------------------------------------
            # RUN CREST
            # ------------------------------------------------

            retained_df, duplicates_df = (
                process_one_structure(

                    xyz_file=xyz_file,

                    results_folder=
                        RESULTS_FOLDER,

                    ring_atoms=
                        config["ring_atoms"],

                    forming_bond=
                        config["forming_bond"],

                    fc=fc,

                    method=
                        batch_method,

                    quick_mode=
                        batch_quick_mode,

                    threads=
                        batch_threads,

                    torsion_tolerance_deg=
                        torsion_tolerance_deg,

                    timeout_seconds=
                        batch_timeout_seconds
                )
            )


        # ----------------------------------------------------
        # Add structure name to retained table.
        # ----------------------------------------------------

        if len(retained_df):

            tmp = retained_df.copy()

            tmp.insert(
                0,
                "structure",
                tag
            )

            all_retained.append(
                tmp
            )


        # ----------------------------------------------------
        # Add structure name to duplicate table.
        # ----------------------------------------------------

        if len(duplicates_df):

            tmp = duplicates_df.copy()

            tmp.insert(
                0,
                "structure",
                tag
            )

            all_duplicates.append(
                tmp
            )


        # ----------------------------------------------------
        # Record successful completion.
        # ----------------------------------------------------

        completed_records.append({

            "structure":
                tag,

            "status":
                "completed",

            "retained_rows":
                len(retained_df),

            "duplicate_rows":
                len(duplicates_df),

            "error":
                "",
        })


        # ----------------------------------------------------
        # CHECKPOINT
        # ----------------------------------------------------

        if all_retained:

            pd.concat(
                all_retained,
                ignore_index=True
            ).to_csv(
                retained_path,
                index=False
            )


        if all_duplicates:

            pd.concat(
                all_duplicates,
                ignore_index=True
            ).to_csv(
                duplicates_path,
                index=False
            )


        pd.DataFrame(
            completed_records
        ).to_csv(
            progress_path,
            index=False
        )


        print()
        print(
            f"COMPLETED: {tag}"
        )

        print(
            f"Retained   : "
            f"{len(retained_df)}"
        )

        print(
            f"Duplicates : "
            f"{len(duplicates_df)}"
        )


    except Exception as exc:

        # ----------------------------------------------------
        # FAILURE
        # ----------------------------------------------------

        failure_record = {

            "structure":
                tag,

            "status":
                "failed",

            "error":
                repr(exc),
        }

        failure_records.append(
            failure_record
        )

        completed_records.append(
            failure_record
        )


        # Save failure checkpoint immediately.
        pd.DataFrame(
            completed_records
        ).to_csv(
            progress_path,
            index=False
        )

        pd.DataFrame(
            failure_records
        ).to_csv(
            failure_path,
            index=False
        )


        print()
        print(
            f"FAILED: {tag}"
        )

        print(
            repr(exc)
        )

        print(
            "The batch will continue "
            "with the next structure."
        )


# ============================================================
# BLOCK 12 — FINAL COMBINED RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("CREATING FINAL COMBINED RESULTS")
print("=" * 80)


# ------------------------------------------------------------
# Combine retained results.
# ------------------------------------------------------------

if all_retained:

    all_retained_df = pd.concat(
        all_retained,
        ignore_index=True
    )

else:

    all_retained_df = pd.DataFrame()


# ------------------------------------------------------------
# Combine duplicate results.
# ------------------------------------------------------------

if all_duplicates:

    all_duplicates_df = pd.concat(
        all_duplicates,
        ignore_index=True
    )

else:

    all_duplicates_df = pd.DataFrame()


# ------------------------------------------------------------
# Failure table.
# ------------------------------------------------------------

failures_df = pd.DataFrame(
    failure_records
)


# ------------------------------------------------------------
# Save final tables.
# ------------------------------------------------------------

all_retained_df.to_csv(
    retained_path,
    index=False
)

all_duplicates_df.to_csv(
    duplicates_path,
    index=False
)

failures_df.to_csv(
    RESULTS_FOLDER
    / "ALL_batch_failures.csv",
    index=False
)


# ------------------------------------------------------------
# Pucker summary.
# ------------------------------------------------------------

if len(all_retained_df):

    summary_df = (

        all_retained_df

        .groupby(
            [
                "structure",
                "classification"
            ],
            dropna=False
        )

        .size()

        .reset_index(
            name="n_retained"
        )

        .sort_values(
            [
                "structure",
                "classification"
            ]
        )

        .reset_index(
            drop=True
        )
    )

    summary_path = (
        RESULTS_FOLDER
        / "SUMMARY_retained_puckers.csv"
    )

    summary_df.to_csv(
        summary_path,
        index=False
    )

    display(summary_df)

else:

    summary_df = pd.DataFrame()


# ------------------------------------------------------------
# Final batch summary.
# ------------------------------------------------------------

print("\n")
print("=" * 80)
print("BATCH FINISHED")
print("=" * 80)

print(
    f"Input structures : "
    f"{len(xyz_files)}"
)

print(
    f"Retained rows     : "
    f"{len(all_retained_df)}"
)

print(
    f"Duplicate rows    : "
    f"{len(all_duplicates_df)}"
)

print(
    f"Failed structures : "
    f"{len(failures_df)}"
)

print(
    f"Results folder    : "
    f"{RESULTS_FOLDER}"
)

print("\nMain output files:")

print(
    f"1. {retained_path}"
)

print(
    f"2. {duplicates_path}"
)

print(
    f"3. {RESULTS_FOLDER / 'ALL_batch_failures.csv'}"
)

print(
    f"4. {RESULTS_FOLDER / 'SUMMARY_retained_puckers.csv'}"
)

print(
    f"5. {RESULTS_FOLDER / 'INPUT_validation.csv'}"
)

print("\nDone.")